# FeatureGraph API starter notebook

Calls the live hosted endpoint and loads the result into a pandas DataFrame.
See `featuregraph-api-usage.md` for the full field reference and what
`smooth_window` does.

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file

BASE_URL = "https://featuregraph-api.fly.dev"
API_KEY = os.environ["FEATUREGRAPH_API_KEY"]  # set in a local .env file

/Users/naziahabib/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Confirm the service is reachable before anything else -- no auth required.
r = requests.get(f"{BASE_URL}/health", timeout=10)
r.raise_for_status()
print(r.json())

{'status': 'ok'}


In [3]:
def run_compiler(dataset: str, smooth_window: int) -> dict:
    """Calls /process and returns the raw JSON response."""
    r = requests.get(
        f"{BASE_URL}/process",
        params={"dataset": dataset, "smooth_window": smooth_window},
        headers={"X-API-Key": API_KEY},
        timeout=60,
    )
    if r.status_code == 401:
        raise ValueError("401 -- check that API_KEY above is set correctly.")
    r.raise_for_status()
    return r.json()

result = run_compiler(dataset="bidmc", smooth_window=100)
print(f"dataset={result['dataset']}  smooth_window={result['smooth_window']}  rows={result['row_count']}")

dataset=bidmc  smooth_window=100  rows=170


In [4]:
df = pd.DataFrame(result["summary"])
df.head()

,subject,respiration_trough_event_id,start_index,rising_duration,falling_duration,peak_index,trough_index,max_raw_signal,max_smooth_signal,min_raw_signal,...,is_complete,duration,period,amplitude_raw,amplitude_smooth,raw_rising_mean_rate,raw_falling_mean_rate,smooth_rising_mean_rate,smooth_falling_mean_rate,temporal_symmetry
0,1,1,101.0,78,158,179.0,101.0,1.0,0.879296,0.140760,...,True,236,NaN,0.429620,0.348336,0.011016,0.005438,0.008932,0.004409,0.661017
1,1,2,337.0,123,159,460.0,337.0,1.0,0.853312,0.019550,...,True,282,281.0,0.490225,0.335298,0.007971,0.006166,0.005452,0.004218,0.872340
2,1,3,619.0,117,170,736.0,619.0,1.0,0.824552,0.001955,...,True,287,276.0,0.499022,0.336151,0.008530,0.005871,0.005746,0.003955,0.815331
3,1,4,906.0,121,172,1027.0,906.0,1.0,0.879062,0.036168,...,True,293,291.0,0.481916,0.371557,0.007966,0.005604,0.006141,0.004320,0.825939
4,1,5,1199.0,122,176,1321.0,1199.0,1.0,0.907600,0.009775,...,True,298,294.0,0.495112,0.387847,0.008117,0.005626,0.006358,0.004407,0.818792


In [5]:
# is_complete=False marks a partial object at the start/end of the recording
# with no real boundary on one side -- drop these before any real analysis,
# the same way the rest of this project treats them.
complete = df[df["is_complete"]]
print(f"{len(df)} total objects, {len(complete)} complete")
complete[["duration", "period", "amplitude_smooth", "temporal_symmetry"]].describe()

170 total objects, 170 complete


,duration,period,amplitude_smooth,temporal_symmetry
count,170.000000,169.000000,170.000000,170.000000
mean,350.023529,350.621302,0.370942,0.875083
std,33.938851,29.854186,0.054804,0.044444
min,39.000000,167.000000,0.001562,0.661017
25%,343.250000,342.000000,0.357193,0.849893
50%,355.000000,355.000000,0.388229,0.873778
75%,366.000000,364.000000,0.401713,0.900179
max,417.000000,446.000000,0.500000,0.989071


## Seeing the effect of `smooth_window`

The window isn't a neutral setting: the same recording at two different
windows can produce very different object counts. Try it directly:

In [6]:
for w in (1, 25, 50, 100, 200):
    result = run_compiler(dataset="bidmc", smooth_window=w)
    print(f"smooth_window={w:>4}  ->  {result['row_count']} objects")

smooth_window=   1  ->  1070 objects
smooth_window=  25  ->  443 objects
smooth_window=  50  ->  272 objects
smooth_window= 100  ->  170 objects
smooth_window= 200  ->  168 objects


In [7]:
result = run_compiler(dataset="bidmc", smooth_window=100)
print(result['signal_column'], result['group_column'])
print(result['summary'][0])

respiration subject
{'subject': 1, 'respiration_trough_event_id': 1, 'start_index': 101.0, 'rising_duration': 78, 'falling_duration': 158, 'peak_index': 179.0, 'trough_index': 101.0, 'max_raw_signal': 1.0, 'max_smooth_signal': 0.8792958499999999, 'min_raw_signal': 0.14076, 'min_smooth_signal': 0.18262390000000003, 'end_index': 337.0, 'is_complete': True, 'duration': 236, 'period': None, 'amplitude_raw': 0.42962, 'amplitude_smooth': 0.3483359749999999, 'raw_rising_mean_rate': 0.011015897435897436, 'raw_falling_mean_rate': 0.005438227848101266, 'smooth_rising_mean_rate': 0.008931691666666665, 'smooth_falling_mean_rate': 0.004409316139240505, 'temporal_symmetry': 0.6610169491525424}


In [8]:
result = run_compiler(dataset="wearable_hr", smooth_window=100)
print(result['signal_column'], result['group_column'])

heart_rate_bpm subject


In [9]:
result = run_compiler(dataset="wearable_hr", smooth_window=100)
print(result['signal_column'], result['group_column'])
print(result['summary'][0])

heart_rate_bpm subject
{'subject': 'S01', 'heart_rate_bpm_trough_event_id': 1, 'start_index': 340.0, 'rising_duration': 189, 'falling_duration': 81, 'peak_index': 529.0, 'trough_index': 340.0, 'max_raw_signal': 75.07, 'max_smooth_signal': 69.66194999999999, 'min_raw_signal': 57.38, 'min_smooth_signal': 59.6362, 'end_index': 610.0, 'is_complete': True, 'duration': 270, 'period': None, 'amplitude_raw': 8.844999999999995, 'amplitude_smooth': 5.012874999999994, 'raw_rising_mean_rate': 0.09359788359788355, 'raw_falling_mean_rate': 0.21839506172839496, 'smooth_rising_mean_rate': 0.05304629629629623, 'smooth_falling_mean_rate': 0.12377469135802455, 'temporal_symmetry': 0.6}


In [10]:
for w in (1, 25, 50, 100, 200):
    result = run_compiler(dataset="wearable_hr", smooth_window=w)
    print(f"smooth_window={w:>4}  ->  {result['row_count']} objects")

smooth_window=   1  ->  79 objects
smooth_window=  25  ->  17 objects
smooth_window=  50  ->  13 objects
smooth_window= 100  ->  7 objects
smooth_window= 200  ->  4 objects
